In [1]:
import pandas as pd

In [ ]:
def csv_merge_etl(file_a, file_b, join_key, output_path):
    print("*** Starting ETL pipeline ***")

    print(f"Extracting data from {file_a} and {file_b}")
    try:
        df_a = pd.read_csv(file_a)
        df_b = pd.read_csv(file_b)
    except FileNotFoundError as e:
        print(f"Extraction error: {e}")
        return None
    
    print(f"Merging on {join_key}")

    merged_df = pd.merge(df_a, df_b, on=join_key, how="left")
    merged_df.fillna("N/A", inplace=True)

    merged_df.to_csv(output_path, index=False)

    print("*** ETL pipeline completed ***")

    customer_totals = merged_df.groupby(["customer_name", "customer_id"])["order_total"].sum().reset_index()
    return customer_totals

In [12]:
csv_merge_etl(
    "../sample_data/sample_customers.csv",
    "../sample_data/sample_orders.csv",
    "customer_id",
    "../output/merged_orders.csv"
)

*** Starting ETL pipeline ***
Extracting data from ../sample_data/sample_customers.csv and ../sample_data/sample_orders.csv
Merging on customer_id
*** ETL pipeline completed ***


,customer_name,customer_id,order_total
0,Anna Wintour,3,20.5
1,Bellamy Blake,1,75.0
2,Blair Waldorf,4,20.0
3,Bonnie Bennett,6,95.0
4,Caroline Forbes,7,5.0
5,Clarke Griffin,2,70.0
6,Harry Potter,8,35.5
7,Hermione Granger,9,20.5
8,Miranda Priestley,5,105.0
9,Ron Weasley,10,25.5


In [ ]:
# Merge without pandas, using only csv module

In [13]:
import csv

In [16]:
customer_lookup = {}

with open("../sample_data/sample_customers.csv", "r", encoding="utf-8") as cust_file:
    cust_reader = csv.DictReader(cust_file)
    cust_headers = cust_reader.fieldnames
    for row in cust_reader:
        customer_lookup[row["customer_id"]] = row

print(customer_lookup)

{'1': {'customer_id': '1', 'customer_name': 'Bellamy Blake'}, '2': {'customer_id': '2', 'customer_name': 'Clarke Griffin'}, '3': {'customer_id': '3', 'customer_name': 'Anna Wintour'}, '4': {'customer_id': '4', 'customer_name': 'Blair Waldorf'}, '5': {'customer_id': '5', 'customer_name': 'Miranda Priestley'}, '6': {'customer_id': '6', 'customer_name': 'Bonnie Bennett'}, '7': {'customer_id': '7', 'customer_name': 'Caroline Forbes'}, '8': {'customer_id': '8', 'customer_name': 'Harry Potter'}, '9': {'customer_id': '9', 'customer_name': 'Hermione Granger'}, '10': {'customer_id': '10', 'customer_name': 'Ron Weasley'}}


In [ ]:
with open("../sample_data/sample_orders.csv", "r", encoding="utf-8") as orders_file, open("../output/joined.csv", "w", newline="", encoding="utf-8") as outfile:
    orders_reader = csv.DictReader(orders_file)
    orders_headers = orders_reader.fieldnames

    output_headers = cust_headers + [h for h in orders_headers if h != "customer_id"]

    writer = csv.DictWriter(outfile, fieldnames=output_headers)
    writer.writeheader()

    for order_row in orders_reader:
        id = order_row["customer_id"]
        customer_row = customer_lookup.get(id, {})

        combined_row = {**customer_row, **order_row}
        writer.writerow(combined_row)